In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
df = pd.read_csv("StudentsPerformance.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1000, 8)


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
#data exploration
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

df.info()

Missing Values:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

Duplicate Rows:
0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64

In [4]:
#remove duplicates
df = df.drop_duplicates()

#Standardize Column Names
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

df.columns

#Create Average Score
df["average_score"] = (
    df["math_score"] +
    df["reading_score"] +
    df["writing_score"]
) / 3

#Create Performance Category
def performance(avg):
    if avg >= 80:
        return "Excellent"
    elif avg >= 60:
        return "Good"
    else:
        return "Needs Improvement"

df["performance"] = df["average_score"].apply(performance)

#Create Pass/Fail Status
df["result"] = df["average_score"].apply(
    lambda x: "Pass" if x >= 40 else "Fail"
)

df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,average_score,performance,result
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667,Good,Pass
1,female,group C,some college,standard,completed,69,90,88,82.333333,Excellent,Pass
2,female,group B,master's degree,standard,none,90,95,93,92.666667,Excellent,Pass
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333,Needs Improvement,Pass
4,male,group C,some college,standard,none,76,78,75,76.333333,Good,Pass


In [5]:
engine = create_engine("sqlite:///student_performance.db")

print("SQLite Database Created Successfully!")

SQLite Database Created Successfully!


In [6]:
df.to_sql(
    name="student_performance",
    con=engine,
    if_exists="replace",
    index=False
)

print("Data Loaded Successfully!")

Data Loaded Successfully!


In [15]:
query = """
SELECT *
FROM student_performance
LIMIT 10
"""

pd.read_sql(query, engine)



,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,average_score,performance,result
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667,Good,Pass
1,female,group C,some college,standard,completed,69,90,88,82.333333,Excellent,Pass
2,female,group B,master's degree,standard,none,90,95,93,92.666667,Excellent,Pass
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333,Needs Improvement,Pass
4,male,group C,some college,standard,none,76,78,75,76.333333,Good,Pass
5,female,group B,associate's degree,standard,none,71,83,78,77.333333,Good,Pass
6,female,group B,some college,standard,completed,88,95,92,91.666667,Excellent,Pass
7,male,group B,some college,free/reduced,none,40,43,39,40.666667,Needs Improvement,Pass
8,male,group D,high school,free/reduced,completed,64,64,67,65.000000,Good,Pass
9,female,group B,high school,free/reduced,none,38,60,50,49.333333,Needs Improvement,Pass


In [10]:
#Average Scores by Gender
query = """
SELECT gender,
ROUND(AVG(average_score),2) AS avg_score
FROM student_performance
GROUP BY gender
"""

pd.read_sql(query, engine)


,gender,avg_score
0,female,69.57
1,male,65.84


In [11]:
#Performance Distribution
query = """
SELECT performance,
COUNT(*) AS total_students
FROM student_performance
GROUP BY performance
"""

pd.read_sql(query, engine)

,performance,total_students
0,Excellent,198
1,Good,517
2,Needs Improvement,285


In [12]:
#Test Preparation Analysis
query = """
SELECT test_preparation_course,
ROUND(AVG(average_score),2) AS avg_score
FROM student_performance
GROUP BY test_preparation_course
"""

pd.read_sql(query, engine)

,test_preparation_course,avg_score
0,completed,72.67
1,none,65.04


In [13]:
#Top 10 Students
query = """
SELECT gender,
race_ethnicity,
average_score,
performance
FROM student_performance
ORDER BY average_score DESC
LIMIT 10
"""

pd.read_sql(query, engine)

,gender,race_ethnicity,average_score,performance
0,female,group E,100.000000,Excellent
1,male,group E,100.000000,Excellent
2,female,group E,100.000000,Excellent
3,female,group E,99.666667,Excellent
4,female,group D,99.000000,Excellent
5,female,group D,99.000000,Excellent
6,female,group C,98.666667,Excellent
7,male,group D,98.666667,Excellent
8,male,group E,97.666667,Excellent
9,female,group E,97.666667,Excellent


In [14]:
#Pass vs Fail Analysis
query = """
SELECT result,
COUNT(*) AS total_students
FROM student_performance
GROUP BY result
"""

pd.read_sql(query, engine)

,result,total_students
0,Fail,30
1,Pass,970


In [18]:
#export cleaned data
df.to_csv(
    "cleaned_student_performance.csv",
    index=False
)

print("Cleaned dataset exported successfully!")

Cleaned dataset exported successfully!


In [17]:
print("ETL Pipeline Completed Successfully")

print("Total Records Processed:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nPerformance Distribution:")
print(df["performance"].value_counts())

ETL Pipeline Completed Successfully
Total Records Processed: 1000

Columns:
['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score', 'average_score', 'performance', 'result']

Performance Distribution:
performance
Good                 517
Needs Improvement    285
Excellent            198
Name: count, dtype: int64
